In [1]:
# Cell 1: Import
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
CUDA available: False


In [2]:
# Cell 2: Khối VGG — xếp chồng 2 lớp Conv 3x3

class VGGBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
    def forward(self, x): return self.block(x)

x = torch.randn(2, 3, 32, 32)
print("VGGBlock output:", VGGBlock(3, 64)(x).shape)

VGGBlock output: torch.Size([2, 64, 16, 16])


In [3]:
# Cell 3: Khối Residual cho 2D và 1D

class ResidualBlock2D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c))
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResidualBlock1D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_c)
        self.conv2 = nn.Conv1d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_c)
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm1d(out_c))
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

print("Residual2D:", ResidualBlock2D(32, 64, stride=2)(torch.randn(2,32,16,16)).shape)
print("Residual1D:", ResidualBlock1D(16, 32, stride=2)(torch.randn(2,16,20)).shape)

Residual2D: torch.Size([2, 64, 8, 8])
Residual1D: torch.Size([2, 32, 10])


In [4]:
# Cell 4: Depthwise Separable Conv (MobileNet-style)

class DepthwiseSeparableConv2D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_c, in_c, 3, stride, 1, groups=in_c, bias=False),
            nn.BatchNorm2d(in_c), nn.ReLU(inplace=True))
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.pointwise(self.depthwise(x))

class DepthwiseSeparableConv1D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv1d(in_c, in_c, 3, stride, 1, groups=in_c, bias=False),
            nn.BatchNorm1d(in_c), nn.ReLU(inplace=True))
        self.pointwise = nn.Sequential(
            nn.Conv1d(in_c, out_c, 1, bias=False),
            nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.pointwise(self.depthwise(x))

# Đếm tham số để so sánh
conv_std  = nn.Conv2d(64, 128, 3, padding=1, bias=False)
conv_dws  = DepthwiseSeparableConv2D(64, 128)
print(f"Conv chuẩn:   {sum(p.numel() for p in conv_std.parameters()):,} tham số")
print(f"DW-Sep:       {sum(p.numel() for p in conv_dws.parameters()):,} tham số")

Conv chuẩn:   73,728 tham số
DW-Sep:       9,152 tham số


In [5]:
# Cell 5: SE-Block 2D và 1D

class SEBlock2D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        s = self.excitation(s).view(b, c, 1, 1)
        return x * s

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.shape
        s = self.squeeze(x).view(b, c)
        s = self.excitation(s).view(b, c, 1)
        return x * s

print("SEBlock2D:", SEBlock2D(32)(torch.randn(2,32,16,16)).shape)
print("SEBlock1D:", SEBlock1D(16)(torch.randn(2,16,20)).shape)

SEBlock2D: torch.Size([2, 32, 16, 16])
SEBlock1D: torch.Size([2, 16, 20])


In [6]:
# Cell 6: CBAM 2D và 1D

class ChannelAttention2D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg + mx).view(b, c, 1, 1)

class SpatialAttention2D(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM2D(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel = ChannelAttention2D(channels, reduction)
        self.spatial = SpatialAttention2D(kernel_size)
    def forward(self, x):
        return self.spatial(self.channel(x))

class ChannelAttention1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _ = x.shape
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg + mx).view(b, c, 1)

class CBAM1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel = ChannelAttention1D(channels, reduction)
    def forward(self, x):
        return self.channel(x)

print("CBAM2D:", CBAM2D(32)(torch.randn(2,32,16,16)).shape)
print("CBAM1D:", CBAM1D(16)(torch.randn(2,16,20)).shape)

CBAM2D: torch.Size([2, 32, 16, 16])
CBAM1D: torch.Size([2, 16, 20])
